In [ ]:
#Data Loading & Preparation Using Python

In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [14]:
# Load dataset
df = pd.read_csv("online_retail_data.csv")

# Basic overview
df.head()


,customer_id,order_date,product_id,category_id,category_name,product_name,quantity,price,payment_method,city,review_score,gender,age
0,13542,12/17/2024,784,10,Electronics,Smartphone,2,373.36,Credit Card,New Oliviaberg,1.00,F,56
1,23188,6/1/2024,682,50,Sports & Outdoors,Soccer Ball,5,299.34,Credit Card,Port Matthew,NaN,M,59
2,55098,2/4/2025,684,50,Sports & Outdoors,Tent,5,23.00,Credit Card,West Sarah,5.00,F,64
3,65208,10/28/2024,204,40,Books & Stationery,Story Book,2,230.11,Bank Transfer,Hernandezburgh,5.00,M,34
4,63872,5/10/2024,202,20,Fashion,Skirt,4,176.72,Credit Card,Jenkinshaven,1.00,F,33


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     1000 non-null   int64  
 1   order_date      1000 non-null   object 
 2   product_id      1000 non-null   int64  
 3   category_id     1000 non-null   int64  
 4   category_name   1000 non-null   object 
 5   product_name    1000 non-null   object 
 6   quantity        1000 non-null   int64  
 7   price           1000 non-null   float64
 8   payment_method  1000 non-null   object 
 9   city            1000 non-null   object 
 10  review_score    799 non-null    float64
 11  gender          897 non-null    object 
 12  age             1000 non-null   int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 101.7+ KB


In [16]:
df.describe()


,customer_id,product_id,category_id,quantity,price,review_score,age
count,"1,000.00","1,000.00","1,000.00","1,000.00","1,000.00",799.00,"1,000.00"
mean,"55,490.72",540.73,30.03,2.95,251.85,3.99,46.38
std,"25,910.19",261.74,14.37,1.41,139.19,1.24,16.57
min,"10,201.00",100.00,10.00,1.00,10.72,1.00,18.00
25%,"33,857.00",311.75,20.00,2.00,128.53,3.00,32.00
50%,"54,619.50",542.50,30.00,3.00,250.22,4.00,47.00
75%,"77,848.50",770.75,40.00,4.00,366.47,5.00,61.00
max,"99,923.00",995.00,50.00,5.00,499.50,5.00,75.00


In [18]:
#finding missing values count
df.isnull().sum()


customer_id         0
order_date          0
product_id          0
category_id         0
category_name       0
product_name        0
quantity            0
price               0
payment_method      0
city                0
review_score      201
gender            103
age                 0
dtype: int64

In [20]:
# Gender: fill missing with 'Unknown'
df['gender'] = df['gender'].fillna('Unknown')

# Review score: keep NaN (no review given)


In [21]:
df['order_date'] = pd.to_datetime(df['order_date'])


In [22]:
#finding Duplicate values count
df.duplicated().sum()

np.int64(0)

In [24]:
# Revenue
df['revenue'] = df['quantity'] * df['price']

# Cost assumed as 70% of revenue
df['cost'] = df['revenue'] * 0.7

# Profit
df['profit'] = df['revenue'] - df['cost']

# Year & Month
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M')


In [25]:
bins = [0, 18, 25, 35, 45, 60, 100]
labels = ['<18', '18-25', '25-35', '35-45', '45-60', '60+']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)


In [ ]:
#Part-2 Exploratory Data Analysis (EDA)

In [26]:
#finding Monthly Sales Trend
monthly_sales = df.groupby('year_month').agg(
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
    orders=('customer_id', 'count')
).reset_index()

monthly_sales


,year_month,total_revenue,total_profit,orders
0,2024-03,"30,620.98","9,186.29",40
1,2024-04,"50,375.40","15,112.62",79
2,2024-05,"61,448.18","18,434.45",92
3,2024-06,"59,127.40","17,738.22",75
4,2024-07,"57,939.86","17,381.96",90
5,2024-08,"79,070.12","23,721.04",90
6,2024-09,"69,323.09","20,796.93",90
7,2024-10,"55,328.98","16,598.69",73
8,2024-11,"55,404.74","16,621.42",75
9,2024-12,"78,432.94","23,529.88",97


In [27]:
# Top 10 products by revenue
top_products = df.groupby('product_name')['revenue'].sum().sort_values(ascending=False).head(10)
top_products


product_name
Smartphone    38,319.26
Notebook      38,027.68
Yoga Mat      37,752.08
Soccer Ball   37,587.30
Tablet        33,581.02
Vase          32,191.58
Laptop        32,000.38
Smartwatch    31,820.10
Headphones    30,789.58
T-shirt       30,718.54
Name: revenue, dtype: float64

In [28]:
# Category performance
category_perf = df.groupby('category_name').agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum'),
    quantity=('quantity', 'sum')
).sort_values(by='revenue', ascending=False)

category_perf


,revenue,profit,quantity
category_name,,,
Electronics,"166,510.34","49,953.10",648
Sports & Outdoors,"154,346.26","46,303.88",625
Books & Stationery,"143,215.52","42,964.66",547
Home & Living,"138,540.15","41,562.05",563
Fashion,"134,714.61","40,414.38",564


In [ ]:
#Customer Segmentation Analysis

In [31]:
#Using age group
age_segment = df.groupby('age_group').agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum'),
    avg_order_value=('revenue', 'mean'),
    customers=('customer_id', 'nunique')
)

age_segment


C:\Users\sutha\AppData\Local\Temp\ipykernel_4812\3191925677.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_segment = df.groupby('age_group').agg(


,revenue,profit,avg_order_value,customers
age_group,,,,
<18,"12,241.90","3,672.57",680.11,18
18-25,"86,536.87","25,961.06",715.18,121
25-35,"126,492.30","37,947.69",739.72,171
35-45,"118,901.40","35,670.42",683.34,174
45-60,"199,282.69","59,784.81",778.45,256
60+,"193,871.72","58,161.52",745.66,260


In [32]:
#by gender segment
gender_segment = df.groupby('gender').agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum'),
    customers=('customer_id', 'nunique')
)

gender_segment


,revenue,profit,customers
gender,,,
F,"331,753.65","99,526.10",440
M,"333,894.19","100,168.26",457
Unknown,"71,679.04","21,503.71",103


In [33]:
city_segment = df.groupby('city').agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum'),
    customers=('customer_id', 'nunique')
).sort_values(by='revenue', ascending=False)

city_segment


,revenue,profit,customers
city,,,
Port Melissaborough,"3,941.29","1,182.39",2
Patriciaville,"3,324.10",997.23,2
Johnsonborough,"3,045.09",913.53,2
East William,"3,011.36",903.41,3
East Christopher,"2,919.20",875.76,2
...,...,...,...
Gonzalezshire,27.65,8.30,1
Port Kimberlymouth,24.00,7.20,1
Williamton,22.21,6.66,1


In [40]:
total_revenue = df['revenue'].sum()
average_order_value = df['revenue'].mean()
total_profit = df['profit'].sum()
customer_count = df['customer_id'].nunique()

total_revenue, average_order_value, total_profit, customer_count


(np.float64(737326.8800000001),
 np.float64(737.3268800000001),
 np.float64(221198.06400000004),
 1000)

In [ ]:
#Business Question Support using Python

In [45]:

#Underperforming Categories
category_perf.sort_values(by='profit').head()


,revenue,profit,quantity
category_name,,,
Fashion,"134,714.61","40,414.38",564
Home & Living,"138,540.15","41,562.05",563
Books & Stationery,"143,215.52","42,964.66",547
Sports & Outdoors,"154,346.26","46,303.88",625
Electronics,"166,510.34","49,953.10",648


In [46]:
#Underperforming Cities
city_segment.sort_values(by='revenue').head()


,revenue,profit,customers
city,,,
Jonathanstad,20.84,6.25,1
North Diane,21.44,6.43,1
Williamton,22.21,6.66,1
Port Kimberlymouth,24.00,7.20,1
Gonzalezshire,27.65,8.30,1


In [47]:
#For Monthly Retention Rate
#first we have to Create customer-month table
customer_month = df[['customer_id', 'year_month']].drop_duplicates()

#then we have to Sort
customer_month = customer_month.sort_values(['customer_id', 'year_month'])

#after short we have to Shift month
customer_month['next_month'] = customer_month.groupby('customer_id')['year_month'].shift(-1)

# Retained if next month purchase exists
customer_month['retained'] = customer_month['year_month'] + 1 == customer_month['next_month']

#finally we got Monthly retention rate
retention = customer_month.groupby('year_month')['retained'].mean().reset_index()
retention


,year_month,retained
0,2024-03,0.00
1,2024-04,0.00
2,2024-05,0.00
3,2024-06,0.00
4,2024-07,0.00
5,2024-08,0.00
6,2024-09,0.00
7,2024-10,0.00
8,2024-11,0.00
9,2024-12,0.00


In [49]:
#Identify Churned Customers (No Purchase in Last 60 Days)
latest_date = df['order_date'].max()

# Last purchase date per customer
last_purchase = df.groupby('customer_id')['order_date'].max().reset_index()

# Days since last purchase
last_purchase['days_since_last_purchase'] = (latest_date - last_purchase['order_date']).dt.days

# Churned customers
churned_customers = last_purchase[last_purchase['days_since_last_purchase'] > 60]

churned_customers.head()


,customer_id,order_date,days_since_last_purchase
0,10201,2024-10-14,156
1,10211,2024-07-23,239
2,10254,2024-09-10,190
3,10299,2024-11-27,112
4,10403,2024-05-03,320


In [52]:
#ans for - Why Sales Dropped in Q2?
# Add quarter
df['quarter'] = df['order_date'].dt.to_period('Q')

quarterly_sales = df.groupby('quarter')['revenue'].sum().reset_index()
quarterly_sales




,quarter,revenue
0,2024Q1,"30,620.98"
1,2024Q2,"170,950.98"
2,2024Q3,"206,333.07"
3,2024Q4,"189,166.66"
4,2025Q1,"140,255.19"


In [53]:
# Compare Q1 vs Q2 by category
q_comparison = df.groupby(['quarter', 'category_name'])['revenue'].sum().reset_index()
q_comparison


,quarter,category_name,revenue
0,2024Q1,Books & Stationery,"6,359.85"
1,2024Q1,Electronics,"5,000.80"
2,2024Q1,Fashion,"3,323.23"
3,2024Q1,Home & Living,"10,019.72"
4,2024Q1,Sports & Outdoors,"5,917.38"
5,2024Q2,Books & Stationery,"37,082.07"
6,2024Q2,Electronics,"35,051.48"
7,2024Q2,Fashion,"32,197.79"
8,2024Q2,Home & Living,"21,316.17"
9,2024Q2,Sports & Outdoors,"45,303.47"
